# SWW to regular-grid tensors — Chowilla River

Convert ANUGA `.sww` outputs and gauge `.tms` files into PyTorch tensors on a regular **`[nx, ny]`** grid.

**Workflow:** define interpolation helpers → define scenario processors → fill elevation NaNs → process and save depth/elevation → process and save discharge.

**Output files (unchanged):** `chowilla_river_processed_data_solution.pt`, `chowilla_river_processed_data_bed.pt`, and `chowilla_river_processed_data_input.pt`.

> Run cells from top to bottom. The existing CUDA-based calculations, parameter values, tensor slicing, and save paths are preserved. The dataset path is set in the processing section below.


## Imports

Libraries used by the processing functions and save steps.

In [1]:
import os
import re

import netCDF4 as nc
import numpy as np
import torch
from scipy.interpolate import griddata
from tqdm import tqdm


## 1. Interpolate SWW depth and elevation

Load the SWW mesh and fields, compute water depth, find nearest mesh nodes on the target grid, and return depth, elevation, and grid coordinates.

In [2]:
def sww_to_depth_tensor_gpu(sww_file, nx=150, ny=150, min_depth_threshold=0.01, device='cuda'):
    """
    Interpolates SWW data to a uniform grid of size (nx, ny) and extracts elevation.
    Target resolution is inferred from the bounding box and the specified nx/ny.
    """

    # 1. Load data
    ds = nc.Dataset(sww_file, 'r')
    x_raw = ds.variables['x'][:]
    y_raw = ds.variables['y'][:]
    elevation_raw = ds.variables['elevation'][:]  
    stage_all_raw = ds.variables['stage'][:]
    ds.close()

    # 2. Calculate Depth (NumPy)
    depth_all_raw = stage_all_raw - elevation_raw
    depth_all_raw = np.nan_to_num(depth_all_raw, nan=0.0)
    depth_all_raw[depth_all_raw < min_depth_threshold] = 0.0
    nt, num_nodes = depth_all_raw.shape

    # 3. Transfer to GPU
    points_gpu = torch.stack([
        torch.from_numpy(x_raw).to(device).float(),
        torch.from_numpy(y_raw).to(device).float()
    ], dim=1)
    
    depth_gpu = torch.from_numpy(depth_all_raw).to(device).float()
    elev_gpu = torch.from_numpy(elevation_raw).to(device).float()

    # 4. Create Uniform Target Grid based on nx and ny
    x_min, x_max = x_raw.min(), x_raw.max()
    y_min, y_max = y_raw.min(), y_raw.max()

    # Using steps=nx and steps=ny to create the linspace directly
    lin_x = torch.linspace(x_min, x_max, steps=nx, device=device)
    lin_y = torch.linspace(y_min, y_max, steps=ny, device=device)
    
    # Generate grid coordinates
    mesh_y, mesh_x = torch.meshgrid(lin_y, lin_x, indexing='ij')
    grid_coords = torch.stack([mesh_x.flatten(), mesh_y.flatten()], dim=1)

    # 5. Find Nearest Neighbors (Chunked)
    num_grid_points = nx * ny
    nearest_indices = torch.zeros(num_grid_points, dtype=torch.long, device=device)
    
    chunk_size = 5000 
    for i in range(0, num_grid_points, chunk_size):
        end_i = min(i + chunk_size, num_grid_points)
        dists = torch.cdist(grid_coords[i:end_i], points_gpu) 
        nearest_indices[i:end_i] = torch.argmin(dists, dim=1)
        
    # 6. Map Depth (all timesteps) and Elevation (static)
    depth_interp_flat = depth_gpu[:, nearest_indices]
    elev_interp_flat = elev_gpu[nearest_indices]
    
    # 7. Reshape and Permute to maintain original [nx, ny, nt] and [nx, ny] output
    tensor_depth = depth_interp_flat.view(nt, ny, nx).permute(2, 1, 0)
    tensor_elev = elev_interp_flat.view(ny, nx).T
    
    grid_x_2d = mesh_x.T # [nx, ny]
    grid_y_2d = mesh_y.T # [nx, ny]
    
    return tensor_depth, tensor_elev, grid_x_2d, grid_y_2d


## 2. Construct Gaussian discharge fields

Read the available gauge `.tms` files, distribute discharge spatially with Gaussian weights, and aggregate the time series into blocks.

In [ ]:
def generate_gaussian_discharge_tensor(tms_dir, grid_x_2d, grid_y_2d, radius=500.0, time_factor=6, device='cuda'):
    """
    Creates a unified spatio-temporal tensor for discharge (Q) using 2D Gaussian kernels
    and Temporal Block Summation with padding to ensure specific output dimensions.
    
    Inputs:
    -------
    - tms_dir (str): Directory containing gauge .tms files.
    - grid_x_2d/grid_y_2d (torch.Tensor): 2D coordinate matrices [nx, ny].
    - gauges (list): Metadata for gauge locations and IDs.
    - radius (float): Spatial influence cutoff.
    - time_factor (int): Factor to reduce the time dimension (e.g., 6).
    - device (str): Target device ('cuda' or 'cpu').

    Outputs:
    
    --------
    - q_tensor (torch.Tensor): Shape [nx, ny, ceil(nt / time_factor)].
    - aggregated_time (np.array): Reduced timeline.
    """



    # chowilla_river gauges
    gauges = [
        {"id": "Bc1_1",  "x": 49530, "y": 11900},
        {"id": "Bc1_2",  "x": 49445, "y": 8250},
        {"id": "Bc1_3",  "x": 49600, "y": 6050},
    ]



    # # dam break gauges
    # CENTER_EASTING  = 3100.0
    # CENTER_NORTHING = 17600.0

    # gauges = [
    #     {"id": "Bc1_1",  "x": CENTER_EASTING, "y": CENTER_NORTHING},
    # ]


    nx, ny = grid_x_2d.shape
    x_offset = 0
    y_offset = 0
    sigma = radius / 2.0 

    # 1. TEMPORAL SYNCHRONIZATION
    master_time = None
    for gauge in gauges:
        usgs_id = gauge['id'].split('_')[-1]
        tms_path = os.path.join(tms_dir, f"{usgs_id}.tms")
        if os.path.exists(tms_path):
            with nc.Dataset(tms_path, 'r') as ds:
                master_time = ds.variables['time'][:].copy()
            break 
            
    if master_time is None:
        raise FileNotFoundError("Could not find any .tms files.")
    
    nt_orig = len(master_time)
    # 2. PADDING LOGIC (To reach 45 steps from 265)
    # Calculate how many extra steps are needed to be divisible by time_factor
    remainder = nt_orig % time_factor
    padding_size = (time_factor - remainder) if remainder > 0 else 0
    nt_padded = nt_orig + padding_size
    nt_new = nt_padded // time_factor # This will be 45 for nt_orig=265, factor=6

    # 3. TENSOR ALLOCATION (Allocate with padded size)
    q_tensor = torch.zeros((nx, ny, nt_padded), device=device)

    # 4. KERNEL GENERATION AND DATA INJECTION
    for gauge in gauges:
        usgs_id = gauge['id'].split('_')[-1]
        tms_path = os.path.join(tms_dir, f"{usgs_id}.tms")
        
        if not os.path.exists(tms_path):
            continue

        with nc.Dataset(tms_path, 'r') as ds:
            q_values = ds.variables['discharge'][:]
        
        q_gpu = torch.from_numpy(q_values).to(device).float()
        local_gx = gauge['x'] - x_offset
        local_gy = gauge['y'] - y_offset

        # 5. SPATIAL WEIGHTING
        dist_sq = (grid_x_2d - local_gx)**2 + (grid_y_2d - local_gy)**2
        weights = torch.exp(-dist_sq / (2 * sigma**2))
        weights[dist_sq > radius**2] = 0 
        
        if weights.sum() > 0:
            weights = weights / weights.sum()

        # 6. INJECTION (Inject original values into the start of the padded tensor)
        # The remaining steps (266-270) remain zero-filled
        q_tensor[..., :nt_orig] += weights.unsqueeze(-1) * q_gpu.unsqueeze(0).unsqueeze(0)

    # 7. TEMPORAL BLOCK SUMMATION
    # Reshape and sum along the block dimension
    q_tensor = q_tensor.view(nx, ny, nt_new, time_factor).sum(dim=-1)

    # 8. TIME AGGREGATION
    # Pad master_time with the last timestamp to match length before reshaping
    if padding_size > 0:
        last_time = master_time[-1]
        time_padding = np.full(padding_size, last_time)
        master_time_padded = np.concatenate([master_time, time_padding])
    else:
        master_time_padded = master_time

    aggregated_time = master_time_padded.reshape(nt_new, time_factor).mean(axis=-1)

    return q_tensor, aggregated_time


## 3. Batch-process depth and elevation

Discover matching SWW scenarios, allocate the output tensors on CPU, and fill them using the GPU interpolation helper.

In [4]:
def process_root_to_depth_tensors(root_dir, nx=328, ny=164, threshold=0.025, q_radius=500.0, time_factor=6, if_UQ=False):
    """
    Processes scenarios by pre-allocating CPU memory to prevent 'cat' memory spikes.
    Static data (grid, elevation, time) is stored only once.
    """
    # 1. First Pass: Find all valid scenarios to determine 'nbatch'
    sww_pattern = re.compile(r"chowilla_river_group_.*_scenario_.*_merged\.sww$")
    scenario_list = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if sww_pattern.match(file):
                tms_dir = os.path.join(root, 'tms_files')
                if os.path.exists(tms_dir):
                    scenario_list.append((os.path.join(root, file), tms_dir))
    
    nbatch = len(scenario_list)
    print(f'nbatch: {nbatch}')
    if nbatch == 0:
        return None

    # 2. Get dimensions from the first scenario to pre-allocate
    first_sww, first_tms = scenario_list[0]
    d_sample, e_sample, gx_sample, gy_sample = sww_to_depth_tensor_gpu(first_sww, nx=nx, ny=ny, min_depth_threshold=threshold)
    q_sample, t_sample = generate_gaussian_discharge_tensor(first_tms, gx_sample, gy_sample, radius=q_radius, time_factor=time_factor)
    
    nx, ny, nt = d_sample.shape
    
    # 3. Pre-allocate dynamic tensors on CPU
    # This allocates the memory ONCE. No more growing lists or 'cat' operations.
    batch_depth = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    batch_elev = torch.zeros((nbatch, nx, ny), dtype=torch.float32)
    # batch_q = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    
    # Store static data only once
    # final_elevation = e_sample.cpu()
    final_gx = gx_sample.cpu()
    final_gy = gy_sample.cpu()
    final_time = torch.from_numpy(t_sample).float()

    print(f"Pre-allocated CPU memory for {nbatch} scenarios ({nx}x{ny}x{nt})")

    # 4. Second Pass: Fill the pre-allocated tensors

    for i, (sww_path, tms_dir) in enumerate(tqdm(scenario_list, desc="Processing Scenarios")):
        try:
            # Process on GPU for speed
            depth, elve, gx, gy = sww_to_depth_tensor_gpu(sww_path, nx=nx, ny=ny, min_depth_threshold=threshold)
            
            # Direct assignment to the CPU tensor
            batch_depth[i] = depth.cpu()
            if if_UQ:
                batch_elev[i] = elve.cpu()
            # batch_q[i] = q_tensor.cpu()
            
            # Explicitly clean up GPU memory after each scenario
            del (depth, 
                # q_tensor, 
                gx, 
                gy)
            
            if if_UQ:
                del elve

            torch.cuda.empty_cache()
            
            # Note: You can remove the old "if (i+1) % 5" print statements 
            # as tqdm provides a much cleaner live update.
                
        except Exception as e:
            # Using tqdm.write prevents the error message from breaking the progress bar
            tqdm.write(f"Error processing scenario {i}: {e}")

    if if_UQ:
        final_elevation = batch_elev
    else:
        final_elevation = e_sample.cpu()

    return {
        "depth": batch_depth,      # [nbatch, nx, ny, nt]
        # "discharge": batch_q,      # [nbatch, nx, ny, nt]
        "elevation": final_elevation, 
        # "grid_x": final_gx, 
        # "grid_y": final_gy,
        # "time": final_time
    }


## 4. Batch-process discharge

Discover the same scenario structure, generate the spatial discharge field for each scenario, and assemble the batch on CPU.

In [5]:
def process_root_to_q_tensors(root_dir, nx=328, ny=164, q_radius=500.0, time_factor=6):
    """
    Processes scenarios by pre-allocating CPU memory for discharge (Q) only.
    Returns the batched Q tensor and the synchronized time array.
    """
    # 1. First Pass: Find all valid scenarios that have a 'tms_files' directory
    sww_pattern = re.compile(r"chowilla_river_group_.*_scenario_.*_merged\.sww$")
    scenario_list = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if sww_pattern.match(file):
                tms_dir = os.path.join(root, 'tms_files')
                if os.path.exists(tms_dir):
                    scenario_list.append((os.path.join(root, file), tms_dir))
    
    nbatch = len(scenario_list)
    if nbatch == 0:
        print("No scenarios with 'tms_files' found.")
        return None

    # 2. Get spatial and temporal dimensions from the first scenario
    # We still need one call to sww_to_depth_tensor to get the grid (gx, gy)
    first_sww, first_tms = scenario_list[0]
    _, _, gx_sample, gy_sample = sww_to_depth_tensor_gpu(first_sww, nx=nx, ny=ny)
    
    # Generate a sample Q to determine the final 'nt' after time_factor aggregation
    q_sample, t_sample = generate_gaussian_discharge_tensor(
        first_tms, gx_sample, gy_sample, radius=q_radius, time_factor=time_factor
    )
    
    nx, ny, nt = q_sample.shape
    print(f'nx, ny, nt: {nx}, {ny}, {nt}')
    # 3. Pre-allocate dynamic Discharge tensor on CPU RAM
    batch_q = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    
    # Store the synchronized time only once
    final_time = torch.from_numpy(t_sample).float()

    print(f"Pre-allocated CPU memory for {nbatch} Discharge scenarios ({nx}x{ny}x{nt})")

    # 4. Second Pass: Fill the pre-allocated Discharge tensor
    for i, (sww_path, tms_dir) in enumerate(tqdm(scenario_list, desc="Generating Q Tensors")):
        try:
            # We need the grid for each scenario to ensure spatial alignment
            _, _, gx, gy = sww_to_depth_tensor_gpu(sww_path, nx=nx, ny=ny)
            
            # Generate Gaussian Q
            q_tensor, _ = generate_gaussian_discharge_tensor(
                tms_dir, gx, gy, radius=q_radius, time_factor=time_factor
            )
            
            # Direct assignment to CPU memory
            batch_q[i] = q_tensor.cpu()
            
            # Explicitly clean up GPU memory
            del q_tensor, gx, gy
            torch.cuda.empty_cache()
                
        except Exception as e:
            tqdm.write(f"Error processing scenario {i}: {e}")

    return {
        "discharge": batch_q, # [nbatch, nx, ny, nt]
        "time": final_time     # [nt]
    }


## 5. Fill missing elevation values

Use nearest-neighbor interpolation to fill NaN values in a 2D elevation grid or a batch of 2D grids.

In [6]:
def fill_edge_nans(bed_tensor: torch.Tensor) -> torch.Tensor:
    """
    Fill NaNs using nearest-neighbor interpolation from existing (non-NaN) pixels.
    Supports:
      - 2D: [nx, ny]
      - 3D: [nbatch, nx, ny]  (process each batch independently)
    """
    if bed_tensor.dim() not in (2, 3):
        raise ValueError(f"Expected 2D or 3D tensor, got shape {tuple(bed_tensor.shape)}")

    device = bed_tensor.device
    out_dtype = bed_tensor.dtype

    # SciPy wants NumPy on CPU; also use float to support NaNs reliably
    bed_np = bed_tensor.detach().to("cpu").numpy()

    def _fill_2d(arr2d: np.ndarray) -> np.ndarray:
        # If there are no NaNs, return fast
        if not np.isnan(arr2d).any():
            return arr2d

        # If everything is NaN, we cannot infer anything
        if np.isnan(arr2d).all():
            return arr2d  # or raise, depending on your preference

        y, x = np.indices(arr2d.shape)
        mask_valid = ~np.isnan(arr2d)

        points_valid = np.stack((y[mask_valid], x[mask_valid]), axis=1)
        values_valid = arr2d[mask_valid]

        points_nan = np.stack((y[~mask_valid], x[~mask_valid]), axis=1)

        filled_values = griddata(
            points_valid,
            values_valid,
            points_nan,
            method="nearest",
        )

        arr2d = arr2d.copy()
        arr2d[~mask_valid] = filled_values
        return arr2d

    if bed_np.ndim == 2:
        filled_np = _fill_2d(bed_np)
        return torch.from_numpy(filled_np).to(device=device, dtype=out_dtype)

    # 3D: process per batch
    filled_np = np.empty_like(bed_np)
    for b in range(bed_np.shape[0]):
        filled_np[b] = _fill_2d(bed_np[b])

    return torch.from_numpy(filled_np).to(device=device, dtype=out_dtype)


## 6. Process depth and elevation

Set the source directory and process SWW scenarios into depth and elevation tensors.

In [ ]:
# ==========================================
# 1. Process and Save Target Tensors (Depth & Elevation)
# ==========================================
# Generate depth and static terrain data

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/chowilla_river_scenario_groups/scenario_groups_1'
if_UQ = False
DATA_depth = process_root_to_depth_tensors(MAIN_PATH, q_radius=2000, time_factor=1, if_UQ=if_UQ)


### Save depth and elevation

Save the depth solution and filled bed elevation, then release memory before discharge processing.

In [ ]:
# Define file paths for the solution (Target) and bed (Static) tensors
solution_file = os.path.join(MAIN_PATH, "chowilla_river_processed_data_solution.pt")
bed_file      = os.path.join(MAIN_PATH, "chowilla_river_processed_data_bed.pt")

bed = fill_edge_nans(DATA_depth['elevation'])

print(f"Saving Target tensors to: {MAIN_PATH}")
torch.save(DATA_depth['depth'][..., :-1][..., :-7], solution_file)
torch.save(bed, bed_file)

# Explicitly clear memory before starting the next heavy process
del DATA_depth, bed
torch.cuda.empty_cache()


## 7. Process discharge

Generate the discharge tensor for all scenarios and define the input tensor output path.

In [7]:
MAIN_PATH = '/storage/group/cxs1024/default/mehdi/chowilla_river_scenario_groups/scenario_groups_1'
if_UQ = False

# ==========================================
# 2. Process and Save Input Tensors (Discharge)
# ==========================================
# Generate the spatio-temporal discharge input

DATA_q = process_root_to_q_tensors(MAIN_PATH, q_radius=10000, time_factor=1)

# Define file path for the input tensor
input_file = os.path.join(MAIN_PATH, "chowilla_river_processed_data_input.pt")


nx, ny, nt: 328, 164, 291
Pre-allocated CPU memory for 500 Discharge scenarios (328x164x291)


Generating Q Tensors: 100%|██████████| 500/500 [03:16<00:00,  2.54it/s]


### Save discharge

Apply the original temporal sampling/cropping, save the discharge tensor, and clear memory.

In [ ]:
print(f"Saving Input tensors to: {MAIN_PATH}")
torch.save(DATA_q['discharge'][..., ::3][..., :-7], input_file)

# Final cleanup
del DATA_q
torch.cuda.empty_cache()

print("✅ Data processing and storage complete.")
